# 环境配置
- conda create -n teach python=3.10
- conda activate teach
- pip install numpy pandas matplotlib scikit-learn

# 遗传算法_鸢尾花

**教学说明（本科生）：**

本节介绍**遗传算法**（Genetic Algorithm, GA）在特征选择问题中的应用。遗传算法是模拟自然界生物进化过程的一种启发式优化算法。

**问题背景：**
在机器学习中，并非所有特征都对分类任务有帮助。我们需要从多个特征中选择一个最优的特征子集，在保证分类性能的同时减少特征维度。

**核心概念：**

#### 🌿 生物学 vs 遗传算法 对照表

| 生物学概念 | 遗传算法中的对应 | 作用 |
|-----------|-----------------|------|
| 种群 (Population) | 候选解集合 | 搜索的"人群" |
| 染色体 (Chromosome) | 一个候选解（0/1串） | 表示特征选择方案 |
| 基因 (Gene) | 染色体上的一个位点 | 表示一个特征是否选中 |
| 适应度 (Fitness) | 交叉验证准确率 | 衡量解的好坏 |
| 选择 (Selection) | 轮盘赌选择 | 优胜劣汰 |
| 交叉 (Crossover) | 基因片段交换 | 产生新解 |
| 变异 (Mutation) | 基因位翻转 | 维持多样性 |
- **种群**：由多个个体（染色体）组成的候选解集合
- **染色体**：用0/1串表示特征选择方案，1表示选中该特征
- **适应度**：用交叉验证准确率衡量特征子集的好坏
- **遗传操作**：选择、交叉、变异三种操作模拟生物进化

**学习目标：**
1. 理解遗传算法的基本原理和流程
2. 掌握如何将特征选择问题编码为染色体
3. 体会进化算法在优化问题中的应用

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

## 读取数据并观察

**教学说明（本科生）：**

鸢尾花（Iris）数据集是机器学习中最经典的数据集之一，包含150个样本，分为3个类别（ Setosa、Versicolour、Virginica），每个样本有4个特征：
- 花萼长度（Sepal Length）
- 花萼宽度（Sepal Width）
- 花瓣长度（Petal Length）
- 花瓣宽度（Petal Width）

**学习目标：**
1. 掌握使用sklearn加载数据集的方法
2. 学习如何观察和理解数据的维度和结构
3. 通过可视化了解数据的分布特性

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("数据维度:", X.shape)
print("特征名:", feature_names)
print("类别名:", target_names)
print("\n前5行数据:")
print(X[:5])

plt.figure(figsize=(6, 5))

for i, label in enumerate(target_names):
    plt.scatter(
        X[y == i, 2],   # petal length
        X[y == i, 3],   # petal width
        label=label
    )

plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.title("Iris Dataset Visualization (Petal)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

# 读取鸢尾花数据
iris = load_iris()

# 构建 DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

# 加入类别编号
df_iris["target"] = iris.target

# 加入类别名称
df_iris["species"] = df_iris["target"].map({
    0: iris.target_names[0],
    1: iris.target_names[1],
    2: iris.target_names[2]
})

# 展示前10行
df_iris.head(10)

## 参数设置 + 辅助函数

**教学说明（本科生）：**

本节定义遗传算法运行所需的关键参数和辅助函数。

**核心参数：**
- **种群大小（POP_SIZE）**：每代个体数量，影响搜索广度
- **染色体长度（CHROM_LENGTH）**：等于特征数量，本例为4
- **最大代数（MAX_GEN）**：算法停止条件之一
- **交叉率（CROSS_RATE）**：个体发生交叉的概率
- **变异率（MUTATION_RATE）**：基因发生变异的概率

**辅助函数：**
- `chromosome_to_features()`：将0/1串转换为特征名称列表
- `fitness()`：计算个体适应度，即交叉验证准确率
- `ensure_valid()`：确保染色体至少有一个特征被选中

**学习重点：**
理解参数设置对算法性能的影响，为后续实验调参打下基础。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
POP_SIZE = 8
CHROM_LENGTH = X.shape[1]   # 4个特征
MAX_GEN = 6
CROSS_RATE = 0.8
MUTATION_RATE = 0.15
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def chromosome_to_features(chromosome):
    return [feature_names[i] for i, gene in enumerate(chromosome) if gene == 1]

def chromosome_to_str(chromosome):
    return "".join(str(x) for x in chromosome)

def ensure_valid(chromosome):
    if sum(chromosome) == 0:
        chromosome[random.randint(0, CHROM_LENGTH - 1)] = 1
    return chromosome

def fitness(chromosome):
    selected_idx = [i for i, gene in enumerate(chromosome) if gene == 1]
    if len(selected_idx) == 0:
        return 0.0

    X_selected = X[:, selected_idx]
    clf = KNeighborsClassifier(n_neighbors=3)
    scores = cross_val_score(clf, X_selected, y, cv=5)
    return scores.mean()

def evaluate(population):
    return [fitness(chromosome) for chromosome in population]

def keep_best(population, fitness_values):
    idx = int(np.argmax(fitness_values))
    return population[idx].copy(), fitness_values[idx], idx

## 初始化种群

**教学说明（本科生）：**

本节实现遗传算法的第一步：初始化种群。

**算法流程（对应伪代码：initialize(P(t))）：**
1. 创建指定数量的随机染色体
2. 确保每个染色体至少有一个"1"（至少选择一个特征）
3. 形成初始种群

**可视化解释：**
种群初始化可以看作在解空间中随机撒点。初始种群的多样性直接影响算法的搜索能力：
- 太少：容易陷入局部最优
- 太多：计算开销增大

**学习目标：**
理解种群初始化在进化算法中的重要性，体会随机性在进化计算中的作用。

> 💡 **教学提示**：初始化就像在黑暗的房间中随机撒豆子——豆子（个体）落在不同的位置，有些离出口（最优解）近，有些离得远。种群多样性越丰富，找到出口的概率越大。

In [ ]:
population = []
for _ in range(POP_SIZE):
    chromosome = [random.randint(0, 1) for _ in range(CHROM_LENGTH)]
    chromosome = ensure_valid(chromosome)
    population.append(chromosome)

print("初始种群：")
for i, chrom in enumerate(population, 1):
    print(f"个体{i}: {chrom} -> {chromosome_to_features(chrom)}")

plt.figure(figsize=(8, 4))
plt.imshow(np.array(population), cmap="YlGnBu", aspect="auto")
plt.colorbar(label="Gene value")
plt.xticks(range(CHROM_LENGTH), feature_names, rotation=20)
plt.yticks(range(len(population)), [f"ind{i+1}" for i in range(len(population))])
plt.title("Initial Population")
plt.tight_layout()
plt.show()

## 评价种群

**教学说明（本科生）：**

本节对种群中的每个个体进行适应度评估，这是遗传算法的"选择压力"来源。

**算法流程（对应伪代码：evaluate(P(t)) 和 keep_best(P(t))）：**
1. 对每个个体，提取选中的特征
2. 使用KNN分类器进行5折交叉验证
3. 以平均准确率作为适应度值
4. 找到当前最优个体

**为什么使用交叉验证？**
交叉验证能更客观地评估模型性能，避免对单一训练/测试划分的过拟合。

**学习重点：**
理解适应度函数在进化算法中的核心作用——它定义了"什么是有好的解"。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
fitness_values = evaluate(population)
best, best_fitness, best_idx = keep_best(population, fitness_values)

print("第0代适应度：")
for i, (chrom, fit) in enumerate(zip(population, fitness_values), 1):
    print(f"个体{i}: {chromosome_to_str(chrom)} -> 特征 {chromosome_to_features(chrom)} -> 适应度 {fit:.4f}")

print("\n当前最优个体:", best)
print("当前最优特征组合:", chromosome_to_features(best))
print("当前最优适应度:", round(best_fitness, 4))

plt.figure(figsize=(9, 4))
plt.bar(range(len(population)), fitness_values)
plt.xticks(range(len(population)), [chromosome_to_str(ch) for ch in population], rotation=30)
plt.ylabel("Fitness (CV Accuracy)")
plt.title("Generation 0 Fitness")
plt.tight_layout()
plt.show()

## 选择阶段

**教学说明（本科生）：**

本节实现遗传算法的选择操作，模拟"优胜劣汰"的自然选择过程。

**算法流程（对应伪代码：selection(P(t))）：**
使用**轮盘赌选择**（Roulette Wheel Selection）：
1. 计算种群总适应度
2. 每个个体被选中的概率 = 个体适应度 / 总适应度
3. 根据概率进行随机选择

**选择策略比较：**
- **轮盘赌**：适应度高的个体有更高概率被选中，但低适应度个体仍有希望
- **精英选择**：直接保留最优个体，防止最优解丢失
- ** tournaments**：随机选择几个个体，从中选最优

**学习重点：**
理解选择操作如何引导搜索方向，体会概率在进化算法中的应用。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

### 轮盘赌选择示例

假设有4个个体，适应度分别为[0.8, 0.4, 0.6, 0.2]：

| 个体 | 适应度 | 占比 | 累积概率 |
|------|--------|------|---------|
| A | 0.8 | 40% | 0~0.40 |
| B | 0.4 | 20% | 0.40~0.60 |
| C | 0.6 | 30% | 0.60~0.90 |
| D | 0.2 | 10% | 0.90~1.00 |

随机数r=0.35 → 选中A ✓
随机数r=0.75 → 选中C ✓

> 💡 **教学提示**：高适应度的个体有更大的扇形区域，但低适应度的个体仍然有可能被选中——这保留了种群的多样性，避免过早陷入局部最优。

In [ ]:
from collections import Counter

total_fitness = sum(fitness_values)
if total_fitness == 0:
    probs = [1 / len(population)] * len(population)
else:
    probs = [f / total_fitness for f in fitness_values]

selected_population = []
selected_indices = []

for _ in range(len(population)):
    r = random.random()
    cumulative = 0
    for i, p in enumerate(probs):
        cumulative += p
        if r <= cumulative:
            selected_population.append(population[i].copy())
            selected_indices.append(i)
            break

print("选择后的种群：")
for i, chrom in enumerate(selected_population, 1):
    print(f"个体{i}: {chrom} -> {chromosome_to_features(chrom)}")

counts = Counter(selected_indices)
selected_counts = [counts.get(i, 0) for i in range(len(population))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(population)), probs)
axes[0].set_xticks(range(len(population)))
axes[0].set_xticklabels([chromosome_to_str(ch) for ch in population], rotation=30)
axes[0].set_title("Selection Probability")
axes[0].set_ylabel("Probability")

axes[1].bar(range(len(population)), selected_counts)
axes[1].set_xticks(range(len(population)))
axes[1].set_xticklabels([chromosome_to_str(ch) for ch in population], rotation=30)
axes[1].set_title("Selected Count")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 交叉阶段

**教学说明（本科生）：**

本节实现遗传算法的交叉操作，模拟生物的基因重组。

**算法流程（对应伪代码：crossover(P(t))）：**
1. 将选中的个体两两配对
2. 以交叉率决定是否发生交叉
3. 随机选择交叉点，交换配对双方的基因片段

**交叉策略：**
- **单点交叉**：在染色体中选择一个位置，交换前后两部分
- **双点交叉**：选择两个交叉点，交换中间部分
- **均匀交叉**：每个基因位独立决定是否交换

**学习重点：**
理解交叉操作如何产生新的解组合，体会"组合创新"在进化中的作用。

### 交叉示例

```
父代1: [1, 0, 1, 0]  ← 选中特征1和3
父代2: [0, 1, 0, 1]  ← 选中特征2和4
交叉点: 位置2
子代1: [1, 0, 0, 1]  ← 继承了父代1的前半和父代2的后半
子代2: [0, 1, 1, 0]  ← 继承了父代2的前半和父代1的后半
```

> 💡 **教学提示**：交叉相当于"基因重组"——将两个较好解的优点结合起来，有可能产生更好的解。就像选育优良品种时，把高产和抗病的植株进行杂交。

In [ ]:
crossed_population = []
crossover_records = []

for i in range(0, len(selected_population), 2):
    parent1 = selected_population[i].copy()

    if i + 1 >= len(selected_population):
        crossed_population.append(parent1)
        crossover_records.append((parent1, None, None, parent1, None))
        break

    parent2 = selected_population[i + 1].copy()

    if random.random() < CROSS_RATE:
        point = random.randint(1, CHROM_LENGTH - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        crossed_population.extend([child1, child2])
        crossover_records.append((parent1, parent2, point, child1, child2))
    else:
        crossed_population.extend([parent1, parent2])
        crossover_records.append((parent1, parent2, None, parent1, parent2))

print("交叉后的种群：")
for i, chrom in enumerate(crossed_population, 1):
    print(f"个体{i}: {chrom} -> {chromosome_to_features(chrom)}")

for k, record in enumerate(crossover_records, 1):
    parent1, parent2, point, child1, child2 = record

    if parent2 is None:
        continue

    print(f"\n第{k}组交叉")
    print("父代1:", chromosome_to_str(parent1))
    print("父代2:", chromosome_to_str(parent2))
    print("交叉点:", point if point is not None else "未交叉")
    print("子代1:", chromosome_to_str(child1))
    print("子代2:", chromosome_to_str(child2))

    fig, ax = plt.subplots(figsize=(8, 2))
    mat = np.array([parent1, parent2, child1, child2])
    ax.imshow(mat, cmap="Pastel1", aspect="auto")
    ax.set_yticks([0, 1, 2, 3])
    ax.set_yticklabels(["parent1", "parent2", "child1", "child2"])
    ax.set_xticks(range(CHROM_LENGTH))
    ax.set_xticklabels(feature_names, rotation=20)

    if point is not None:
        ax.axvline(point - 0.5, color="red", linestyle="--", linewidth=2)

    ax.set_title(f"Crossover Group {k}")
    plt.tight_layout()
    plt.show()

## 变异阶段

**教学说明（本科生）：**

本节实现遗传算法的变异操作，模拟生物的基因突变。

**算法流程（对应伪代码：mutation(P(t))）：**
1. 遍历染色体的每个基因位
2. 以变异率决定是否翻转该位（0→1 或 1→0）
3. 确保变异后染色体仍然有效（至少有一个1）

**变异的作用：**
- **维持多样性**：防止种群过早收敛到局部最优
- **探索新区域**：在解空间中探索新的区域
- **理论保证**：理论上，足够的变异能让算法找到全局最优

**学习重点：**
理解变异在保持种群多样性中的关键作用，体会小概率事件在进化中的重要意义。

### 为什么变异率不能太大？

- **变异率过大**（如0.5）：种群变成随机搜索，进化失去方向
- **变异率过小**（如0.001）：种群可能过早收敛，无法跳出局部最优
- **合适的变异率**（如0.05~0.15）：在探索和利用之间取得平衡

> 💡 **教学提示**：变异率就像烹饪中的盐——放少了没味道，放多了菜就毁了。

In [ ]:
mutated_population = []
mutation_records = []

for chromosome in crossed_population:
    original = chromosome.copy()
    mutated = chromosome.copy()
    mutated_positions = []

    for i in range(CHROM_LENGTH):
        if random.random() < MUTATION_RATE:
            mutated[i] = 1 - mutated[i]
            mutated_positions.append(i)

    mutated = ensure_valid(mutated)
    mutated_population.append(mutated)
    mutation_records.append((original, mutated, mutated_positions))

print("变异后的种群：")
for i, chrom in enumerate(mutated_population, 1):
    print(f"个体{i}: {chrom} -> {chromosome_to_features(chrom)}")

for i, (before, after, positions) in enumerate(mutation_records, 1):
    if len(positions) == 0:
        continue

    print(f"\n个体{i} 发生变异，位置: {positions}")
    print("变异前:", chromosome_to_str(before))
    print("变异后:", chromosome_to_str(after))

    fig, ax = plt.subplots(figsize=(7, 1.8))
    mat = np.array([before, after])
    ax.imshow(mat, cmap="coolwarm", aspect="auto")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["before", "after"])
    ax.set_xticks(range(CHROM_LENGTH))
    ax.set_xticklabels(feature_names, rotation=20)

    for p in positions:
        ax.add_patch(plt.Rectangle((p - 0.5, -0.5), 1, 2, fill=False, edgecolor="yellow", linewidth=3))

    ax.set_title(f"Mutation of individual {i}")
    plt.tight_layout()
    plt.show()

## 第1代重新评价，并和第0代比较

**教学说明（本科生）：**

本节对新一代种群进行评价，并与历史最优解比较。

**算法流程（对应伪代码：evaluate(P(t)) 与 replace(Best)）：**
1. 评估新一代种群的适应度
2. 找到新一代的最优个体
3. 与历史最优比较，如果更优则更新

**进化机制：**
遗传算法采用**精英策略**（Elitism）：确保历史最优解不会在进化中丢失。这种机制保证了算法的收敛性。

**学习重点：**
理解种群进化过程中的信息保留机制，体会如何平衡探索（exploration）和利用（exploitation）。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
fitness_values_1 = evaluate(mutated_population)
best_1, best_fitness_1, best_idx_1 = keep_best(mutated_population, fitness_values_1)

print("第1代适应度：")
for i, (chrom, fit) in enumerate(zip(mutated_population, fitness_values_1), 1):
    print(f"个体{i}: {chromosome_to_str(chrom)} -> 特征 {chromosome_to_features(chrom)} -> 适应度 {fit:.4f}")

print("\n第0代最优:", round(best_fitness, 4), best)
print("第1代最优:", round(best_fitness_1, 4), best_1)

if best_fitness_1 > best_fitness:
    print("结果：第1代最优优于历史最优，用第1代最优替换 Best")
else:
    print("结果：历史最优保持不变")

plt.figure(figsize=(9, 4))
plt.bar(range(len(mutated_population)), fitness_values_1)
plt.xticks(range(len(mutated_population)), [chromosome_to_str(ch) for ch in mutated_population], rotation=30)
plt.ylabel("Fitness (CV Accuracy)")
plt.title("Generation 1 Fitness")
plt.tight_layout()
plt.show()

## 完整运行GA

**教学说明（本科生）：**

本节将遗传算法的各个组成部分整合起来，完整运行进化过程。

**完整流程：**
```
初始化种群 → 评价 → 选择 → 交叉 → 变异 → 评价 → 替换 → 循环
```

**算法终止条件：**
- 达到最大代数
- 适应度收敛（变化很小）
- 达到满意的解

**学习重点：**
理解遗传算法的整体流程，体会各个操作如何协同工作，观察适应度随代数的变化趋势。

> 💡 **教学提示**：初始化就像在黑暗的房间中随机撒豆子——豆子（个体）落在不同的位置，有些离出口（最优解）近，有些离得远。种群多样性越丰富，找到出口的概率越大。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

### GA完整流程总结

```
1. 初始化种群  →  2. 评价适应度  →  3. 选择（轮盘赌）
    ↑                              ↓
6. 替换更新  ←  5. 变异  ←  4. 交叉
```

> 💡 **教学提示**：观察每代的种群热力图，可以看到染色体从随机逐渐演化出某种模式——这就是进化在起作用！

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

history_best = []
history_avg = []

t = 0

# initialize
population = []
for _ in range(POP_SIZE):
    chromosome = [random.randint(0, 1) for _ in range(CHROM_LENGTH)]
    chromosome = ensure_valid(chromosome)
    population.append(chromosome)

# evaluate
fitness_values = evaluate(population)
global_best, global_best_fitness, _ = keep_best(population, fitness_values)

history_best.append(global_best_fitness)
history_avg.append(np.mean(fitness_values))

print(f"===== 第 {t} 代 =====")
plt.figure(figsize=(8, 3.5))
plt.imshow(np.array(population), cmap="YlGnBu", aspect="auto")
plt.xticks(range(CHROM_LENGTH), feature_names, rotation=20)
plt.yticks(range(len(population)), [f"ind{i+1}" for i in range(len(population))])
plt.title(f"Generation {t} Population")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.5))
plt.bar(range(len(population)), fitness_values)
plt.xticks(range(len(population)), [chromosome_to_str(ch) for ch in population], rotation=30)
plt.title(f"Generation {t} Fitness")
plt.tight_layout()
plt.show()

while t < MAX_GEN:
    # selection
    total_fitness = sum(fitness_values)
    if total_fitness == 0:
        probs = [1 / len(population)] * len(population)
    else:
        probs = [f / total_fitness for f in fitness_values]

    selected_population = []
    for _ in range(len(population)):
        r = random.random()
        cumulative = 0
        for i, p in enumerate(probs):
            cumulative += p
            if r <= cumulative:
                selected_population.append(population[i].copy())
                break

    # crossover
    crossed_population = []
    for i in range(0, len(selected_population), 2):
        parent1 = selected_population[i].copy()

        if i + 1 >= len(selected_population):
            crossed_population.append(parent1)
            break

        parent2 = selected_population[i + 1].copy()

        if random.random() < CROSS_RATE:
            point = random.randint(1, CHROM_LENGTH - 1)
            child1 = parent1[:point] + parent2[point:]
            child2 = parent2[:point] + parent1[point:]
            crossed_population.extend([child1, child2])
        else:
            crossed_population.extend([parent1, parent2])

    # mutation
    mutated_population = []
    for chromosome in crossed_population:
        mutated = chromosome.copy()
        for i in range(CHROM_LENGTH):
            if random.random() < MUTATION_RATE:
                mutated[i] = 1 - mutated[i]
        mutated = ensure_valid(mutated)
        mutated_population.append(mutated)

    # update generation
    t += 1
    population = mutated_population
    fitness_values = evaluate(population)

    current_best, current_best_fitness, _ = keep_best(population, fitness_values)

    if current_best_fitness > global_best_fitness:
        global_best = current_best.copy()
        global_best_fitness = current_best_fitness

    history_best.append(global_best_fitness)
    history_avg.append(np.mean(fitness_values))

    print(f"===== 第 {t} 代 =====")
    print("本代最优个体:", current_best, "->", chromosome_to_features(current_best))
    print("本代最优适应度:", round(current_best_fitness, 4))
    print("历史最优适应度:", round(global_best_fitness, 4))

    plt.figure(figsize=(8, 3.5))
    plt.imshow(np.array(population), cmap="YlGnBu", aspect="auto")
    plt.xticks(range(CHROM_LENGTH), feature_names, rotation=20)
    plt.yticks(range(len(population)), [f"ind{i+1}" for i in range(len(population))])
    plt.title(f"Generation {t} Population")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 3.5))
    plt.bar(range(len(population)), fitness_values)
    plt.xticks(range(len(population)), [chromosome_to_str(ch) for ch in population], rotation=30)
    plt.title(f"Generation {t} Fitness")
    plt.tight_layout()
    plt.show()

print("最终历史最优解:", global_best)
print("最终历史最优特征组合:", chromosome_to_features(global_best))
print("最终历史最优适应度:", round(global_best_fitness, 4))

## 画适应度进化曲线

**教学说明（本科生）：**

本节可视化适应度随代数变化的曲线，这是评估进化算法性能的重要工具。

**学习重点：**
- **收敛性**：曲线是否趋于平稳
- **收敛速度**：达到较优解所需的代数
- **稳定性**：多次运行的结果是否稳定

**典型曲线特征：**
- 初始阶段：快速上升（种群快速适应）
- 中期阶段：缓慢上升（局部搜索）
- 后期阶段：趋于平稳（收敛）

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(history_best)), history_best, marker='o', label='Global Best Fitness')
plt.plot(range(len(history_avg)), history_avg, marker='s', label='Average Fitness')
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title("GA Fitness Evolution")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 粒子群算法_wine

**教学说明（本科生）：**

本节介绍**粒子群优化算法**（Particle Swarm Optimization, PSO）在葡萄酒数据集特征选择中的应用。

**粒子群算法简介：**
粒子群算法是另一种著名的群体智能算法，模拟鸟群觅食行为。每个粒子在解空间中飞行，通过跟踪个体最优和全局最优来调整自己的速度和位置。

**核心概念：**
- **粒子**：代表一个候选解（特征选择方案）
- **速度**：决定粒子下一次移动的方向和距离
- **个体最优（pBest）**：粒子自身 history 中最好的位置
- **全局最优（gBest）**：整个种群 history 中最好的位置

**与遗传算法的异同：**
| 特征 | 遗传算法 | 粒子群算法 |
|------|---------|-----------|
| 模拟对象 | 生物进化 | 鸟群觅食 |
| 操作 | 选择、交叉、变异 | 速度更新、位置更新 |
| 信息共享 | 间接（通过种群） | 直接（pBest/gBest） |
| 适用问题 | 离散/连续 | 连续为主 |

**学习目标：**
1. 理解粒子群算法的基本原理
2. 对比不同群体智能算法的异同
3. 掌握PSO在特征选择中的应用

### GA vs PSO 对比

| 比较维度 | 遗传算法 (GA) | 粒子群算法 (PSO) |
|---------|-------------|-----------------|
| 模拟对象 | 生物进化（自然选择） | 鸟群觅食（群体智能） |
| 核心操作 | 选择、交叉、变异 | 速度更新、位置更新 |
| 信息共享 | 间接（通过种群基因池） | 直接（pBest + gBest指导） |
| 收敛速度 | 较慢 | 较快 |
| 局部最优 | 较不容易陷入 | 较容易早熟收敛 |
| 参数数量 | 较多（种群大小/交叉率/变异率） | 较少（w/c1/c2） |

## 读取wine数据

**教学说明（本科生）：**

葡萄酒数据集（Wine Dataset）是另一个经典的分类数据集，包含178个样本，分为3个类别，每个样本有13个化学特征：

**13个特征包括：**
- Alcohol, Malic acid, Ash, Alcalinity of ash
- Magnesium, Total phenols, Flavanoids, Nonflavanoid phenols
- Proanthocyanins, Color intensity, Hue, OD280/OD315 of diluted wines, Proline

**学习目标：**
1. 学习处理高维数据的方法
2. 了解不同数据集的特征结构
3. 掌握数据可视化技巧

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA

wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print("数据维度:", X.shape)
print("类别名:", target_names)
print("特征数:", len(feature_names))
print("前13个特征名:", feature_names[:13])
print("\n前5行数据:")
print(X[:5])

In [ ]:
import pandas as pd

# 读取数据
wine = load_wine()

# 构建 DataFrame
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)

# 加入类别编号
df_wine["target"] = wine.target

# 加入类别名称
df_wine["class"] = df_wine["target"].map({
    0: wine.target_names[0],
    1: wine.target_names[1],
    2: wine.target_names[2]
})

print("数据维度:", df_wine.shape)
df_wine

In [ ]:
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

# 找到对应索引
x_index = feature_names.index('flavanoids')
y_index = feature_names.index('color_intensity')

plt.figure(figsize=(6, 5))

for i, label in enumerate(target_names):
    plt.scatter(
        X[y == i, x_index],
        X[y == i, y_index],
        label=label
    )

plt.xlabel("Flavanoids")
plt.ylabel("Color Intensity")
plt.title("Wine Dataset Visualization")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 5))
for i, label in enumerate(target_names):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=label)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine Dataset PCA")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 设置pso参数

**教学说明（本科生）：**

本节定义粒子群算法的关键参数。

**核心参数：**
- **粒子数（N）**：种群中粒子的数量，影响搜索能力和计算开销
- **维度（DIM）**：等于特征数量，本例为13
- **最大迭代次数（MAX_ITER）**：算法停止条件之一
- **惯性权重（w）**：平衡全局探索和局部开发，通常取0.7-0.9
- **个体学习因子（c1）**：个体最优对速度的影响
- **群体学习因子（c2）**：全局最优对速度的影响
- **位置范围**：粒子位置的上下界
- **阈值**：将连续位置转换为0/1选择的阈值

**参数调优经验：**
- w 大：全局搜索能力强，但收敛慢
- w 小：局部搜索强，但易陷入局部最优
- c1 大：个体探索强
- c2 大：群体共识强

### 速度更新公式解析

`v[i] = w·v[i] + c1·r1·(pBest[i] - x[i]) + c2·r2·(gBest - x[i])`

| 项 | 名称 | 含义 | 教学类比 |
|----|------|------|---------|
| w·v[i] | 惯性项 | 保持当前飞行方向 | "继续沿当前方向飞" |
| c1·r1·(pBest-x) | 个体认知项 | 向自己历史上最好的位置移动 | "回忆自己过去的最佳经验" |
| c2·r2·(gBest-x) | 社会项 | 向群体中最好的位置移动 | "向学霸学习" |

> 💡 **教学提示**：速度更新就像是个人决策：既参考自己的经验（pBest），也参考他人的经验（gBest），同时保留一定的惯性（w）避免频繁转向。

In [ ]:
# 粒子群参数
N = 10                 # 粒子数
DIM = X.shape[1]       # 13个特征
MAX_ITER = 12          # 最大迭代次数

w = 0.729                # 惯性权重
c1 = 1.42             # 个体学习因子
c2 = 1.42            # 群体学习因子

LOWER = 0.0
UPPER = 1.0
THRESHOLD = 0.5

np.random.seed(42)

print("粒子数:", N)
print("特征维度:", DIM)
print("最大迭代次数:", MAX_ITER)

def position_to_binary(position, threshold=THRESHOLD):
    """把连续粒子位置转成0/1特征选择"""
    binary = (position > threshold).astype(int)

    # 至少保留一个特征
    if binary.sum() == 0:
        binary[np.argmax(position)] = 1

    return binary


def binary_to_feature_names(binary):
    return [feature_names[i] for i, b in enumerate(binary) if b == 1]


def fitness(position):
    """
    适应度函数：
    把粒子位置转成特征选择方案，再用KNN分类准确率作为适应度
    """
    binary = position_to_binary(position)
    selected_idx = np.where(binary == 1)[0]

    X_selected = X[:, selected_idx]
    clf = KNeighborsClassifier(n_neighbors=3)
    score = cross_val_score(clf, X_selected, y, cv=5).mean()
    return score

## 随机初始化每个粒子

**教学说明（本科生）：**

本节初始化粒子的位置和速度。

**初始化策略：**
- **位置**：在[LOWER, UPPER]范围内随机初始化
- **速度**：在[-0.2, 0.2]范围内随机初始化（较小的初始速度有助于精细搜索）

**位置到选择的转换：**
由于特征选择是离散问题，我们将连续位置通过阈值转换为0/1：
- position[i] > threshold → 选中特征 i
- position[i] ≤ threshold → 不选中特征 i

**学习重点：**
理解连续空间搜索如何应用于离散优化问题，体会问题转换的思想。

In [ ]:
positions = np.random.uniform(LOWER, UPPER, size=(N, DIM))
velocities = np.random.uniform(-0.2, 0.2, size=(N, DIM))

print("初始粒子位置（前3个粒子）：")
print(np.round(positions[:3], 3))

print("\n初始粒子速度（前3个粒子）：")
print(np.round(velocities[:3], 3))

plt.figure(figsize=(10, 4))
plt.imshow(positions, cmap="YlGnBu", aspect="auto")
plt.colorbar(label="Position Value")
plt.xticks(range(DIM), feature_names, rotation=60)
plt.yticks(range(N), [f"P{i+1}" for i in range(N)])
plt.title("Initial Particle Positions")
plt.tight_layout()
plt.show()

## 评估每个粒子并初始化 pBest 与 gBest

**教学说明（本科生）：**

本节初始化粒子的个体最优（pBest）和全局最优（gBest）。

**初始化过程：**
1. 对每个粒子，评估其适应度
2. 将初始位置设为 pBest
3. 在所有 pBest 中找到最好的一个设为 gBest

**对应伪代码：**
```
Evaluate particle i and set pBest_i = X_i
gBest = argmax {pBest_i for all i}
```

**学习重点：**
理解 pBest 和 gBest 在PSO中的作用：
- pBest：记录个体的最好经验
- gBest：记录种群的最好经验，指导整个种群的搜索方向

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
pbest_positions = positions.copy()
pbest_values = np.array([fitness(pos) for pos in positions])

gbest_index = np.argmax(pbest_values)
gbest_position = pbest_positions[gbest_index].copy()
gbest_value = pbest_values[gbest_index]

print("初始化后的粒子评价：\n")
for i in range(N):
    binary = position_to_binary(positions[i])
    print(f"粒子{i+1}")
    print("  二值化:", binary.tolist())
    print("  选中特征数:", int(binary.sum()))
    print("  特征组合:", binary_to_feature_names(binary))
    print("  适应度:", round(pbest_values[i], 4))
    print()

print("初始全局最优 gBest:")
print("适应度:", round(gbest_value, 4))
print("二值化:", position_to_binary(gbest_position).tolist())
print("特征组合:", binary_to_feature_names(position_to_binary(gbest_position)))

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(range(N), pbest_values)
plt.xticks(range(N), [f"P{i+1}" for i in range(N)])
plt.ylabel("Fitness (Accuracy)")
plt.title("Initial Fitness of Particles")
plt.tight_layout()
plt.show()

## 定义一次迭代更新（速度和位置）

**教学说明（本科生）：**

本节实现PSO的核心——速度和位置更新公式。

**PSO更新公式：**
```
v[i] = w * v[i] + c1 * r1 * (pBest[i] - x[i]) + c2 * r2 * (gBest - x[i])
x[i] = x[i] + v[i]
```

**公式解释：**
- **第一项**（w * v[i]）：惯性项，保持原有运动方向
- **第二项**（c1 * r1 * (pBest[i] - x[i])）：个体认知项，向自身最优移动
- **第三项**（c2 * r2 * (gBest - x[i])）：社会项，向全局最优移动

**对应伪代码：**
```
Update the velocity and position of particle i
Evaluate particle i
if fit(Xi) better than fit(pBesti), then update pBesti
if fit(pBesti) better than fit(gBest), then update gBest
```

**学习重点：**
理解速度更新公式的物理意义，体会"个体经验"和"群体智慧"的结合。

In [ ]:
def pso_one_iteration(positions, velocities, pbest_positions, pbest_values, gbest_position, gbest_value):
    positions = positions.copy()
    velocities = velocities.copy()
    pbest_positions = pbest_positions.copy()
    pbest_values = pbest_values.copy()

    for i in range(N):
        r1 = np.random.rand(DIM)
        r2 = np.random.rand(DIM)

        # 更新速度
        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest_positions[i] - positions[i])
            + c2 * r2 * (gbest_position - positions[i])
        )

        # 更新位置
        positions[i] = positions[i] + velocities[i]
        positions[i] = np.clip(positions[i], LOWER, UPPER)

        # 计算当前适应度
        current_value = fitness(positions[i])

        # 更新个体最优
        if current_value > pbest_values[i]:
            pbest_values[i] = current_value
            pbest_positions[i] = positions[i].copy()

        # 更新全局最优
        if pbest_values[i] > gbest_value:
            gbest_value = pbest_values[i]
            gbest_position = pbest_positions[i].copy()

    return positions, velocities, pbest_positions, pbest_values, gbest_position, gbest_value

## 1 次迭代并观察变化

**教学说明（本科生）：**

本节演示一次迭代后粒子位置和最优解的变化。

**观察要点：**
1. 粒子位置是否向最优解方向移动
2. pBest 是否更新
3. gBest 是否更新

**可视化解释：**
通过比较迭代前后的热力图，可以直观看到粒子群体的移动趋势。

**学习重点：**
理解迭代过程的微观变化，体会算法如何逐步改进解的质量。

In [ ]:
new_positions, new_velocities, new_pbest_positions, new_pbest_values, new_gbest_position, new_gbest_value = pso_one_iteration(
    positions, velocities, pbest_positions, pbest_values, gbest_position, gbest_value
)

print("第1次迭代后：\n")
for i in range(N):
    binary = position_to_binary(new_positions[i])
    print(f"粒子{i+1}")
    print("  二值化:", binary.tolist())
    print("  选中特征数:", int(binary.sum()))
    print("  特征组合:", binary_to_feature_names(binary))
    print("  当前pBest适应度:", round(new_pbest_values[i], 4))
    print()

print("更新后的 gBest:")
print("适应度:", round(new_gbest_value, 4))
print("二值化:", position_to_binary(new_gbest_position).tolist())
print("特征组合:", binary_to_feature_names(position_to_binary(new_gbest_position)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(positions, cmap="YlGnBu", aspect="auto")
axes[0].set_title("Before Update")
axes[0].set_xticks(range(DIM))
axes[0].set_xticklabels(feature_names, rotation=60)
axes[0].set_yticks(range(N))
axes[0].set_yticklabels([f"P{i+1}" for i in range(N)])

axes[1].imshow(new_positions, cmap="YlGnBu", aspect="auto")
axes[1].set_title("After 1 Iteration")
axes[1].set_xticks(range(DIM))
axes[1].set_xticklabels(feature_names, rotation=60)
axes[1].set_yticks(range(N))
axes[1].set_yticklabels([f"P{i+1}" for i in range(N)])

plt.tight_layout()
plt.show()

## 完整运行 PSO 主循环

**教学说明（本科生）：**

本节完整运行PSO算法，观察多次迭代后的收敛效果。

**完整流程：**
```
初始化 → 评估 → 迭代更新 → 评估 → 记录 → 循环
```

**收敛曲线分析：**
- 初期快速上升：粒子快速找到较优区域
- 中期缓慢上升：在局部最优附近精细搜索
- 后期趋于平稳：算法收敛

**学习重点：**
理解PSO的收敛特性，掌握算法参数对收敛速度和质量的影响。

### 如何解读收敛曲线

- **快速下降期**：粒子快速找到较优区域，曲线陡峭
- **缓慢下降期**：在局部最优附近精细搜索，曲线平缓
- **平台期**：算法收敛，曲线趋于稳定

如果曲线一直不下降，说明参数设置可能有问题（如w太小导致早熟，或w太大导致不收敛）。

In [ ]:
# 重新初始化，保证完整实验从头开始
np.random.seed(42)

positions = np.random.uniform(LOWER, UPPER, size=(N, DIM))
velocities = np.random.uniform(-0.2, 0.2, size=(N, DIM))

pbest_positions = positions.copy()
pbest_values = np.array([fitness(pos) for pos in positions])

gbest_index = np.argmax(pbest_values)
gbest_position = pbest_positions[gbest_index].copy()
gbest_value = pbest_values[gbest_index]

history_gbest = [gbest_value]
history_feature_count = [position_to_binary(gbest_position).sum()]

print("初始 gBest =", round(gbest_value, 4), binary_to_feature_names(position_to_binary(gbest_position)))

for t in range(MAX_ITER):
    positions, velocities, pbest_positions, pbest_values, gbest_position, gbest_value = pso_one_iteration(
        positions, velocities, pbest_positions, pbest_values, gbest_position, gbest_value
    )

    history_gbest.append(gbest_value)
    history_feature_count.append(position_to_binary(gbest_position).sum())

    print(f"第 {t+1} 代:")
    print("  gBest适应度:", round(gbest_value, 4))
    print("  gBest二值化:", position_to_binary(gbest_position).tolist())
    print("  选中特征数:", int(position_to_binary(gbest_position).sum()))
    print("  gBest特征组合:", binary_to_feature_names(position_to_binary(gbest_position)))
    print()

## 收敛曲线

**教学说明（本科生）：**

可视化全局最优适应度随迭代次数的变化。

**学习重点：**
- **收敛速度**：达到稳定值所需的迭代次数
- **最终性能**：收敛到的最优值
- **稳定性**：曲线是否平滑

**与遗传算法对比：**
- PSO通常收敛更快，但可能早熟收敛
- GA收敛较慢但全局搜索能力更强
- 两者都是启发式算法，没有 guarantees of optimality

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

### 如何解读收敛曲线

- **快速下降期**：粒子快速找到较优区域，曲线陡峭
- **缓慢下降期**：在局部最优附近精细搜索，曲线平缓
- **平台期**：算法收敛，曲线趋于稳定

如果曲线一直不下降，说明参数设置可能有问题（如w太小导致早熟，或w太大导致不收敛）。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(history_gbest)), history_gbest, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Global Best Fitness")
plt.title("PSO Convergence on Wine Feature Selection")
plt.grid(True)
plt.tight_layout()
plt.show()

## gBest 的特征数量变化

**教学说明（本科生）：**

可视化最优解的特征数量随迭代的变化。

**观察要点：**
- 特征数量是否趋于稳定
- 最终选择的特征数量是多少
- 特征选择的稀疏性

**特征选择的意义：**
- **降维**：减少特征数量，降低计算开销
- **去噪**：去除不相关或冗余特征
- **可解释性**：更少的特征更容易理解

**学习重点：**
理解特征选择与模型性能之间的权衡关系。

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(len(history_feature_count)), history_feature_count, marker='s')
plt.xlabel("Iteration")
plt.ylabel("Number of Selected Features")
plt.title("Selected Feature Count of gBest")
plt.grid(True)
plt.tight_layout()
plt.show()

## 最终最优特征组合

**教学说明（本科生）：**

本节展示PSO找到的最优特征选择方案。

**输出信息：**
1. 最终最优适应度（分类准确率）
2. 最终选中的特征索引和名称
3. 选中特征的数量

**学习重点：**
理解如何从进化算法中提取最终解，并评估其性能。

> 💡 **教学提示**：适应度函数是进化的"指挥棒"——它定义了什么样的解是"好"的。如果适应度函数设计不当，进化就会朝错误的方向发展。

In [ ]:
final_binary = position_to_binary(gbest_position)
selected_idx = np.where(final_binary == 1)[0]

print("最终最优适应度:", round(gbest_value, 4))
print("最终最优二值化:", final_binary.tolist())
print("最终选中特征数:", int(final_binary.sum()))
print("最终选中特征:")
for i in selected_idx:
    print("-", feature_names[i])

plt.figure(figsize=(10, 4))
plt.bar(feature_names, final_binary)
plt.ylabel("Selected (1=yes, 0=no)")
plt.title("Best Feature Selection by PSO on Wine Dataset")
plt.xticks(rotation=60)
plt.ylim(0, 1.2)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 比较"全部特征"与"PSO选中特征"的分类效果

**教学说明（本科生）：**

本节对比使用全部特征和PSO选中特征的分类性能。

**评估指标：**
1. **准确率**：分类正确的样本比例
2. **特征数量**：选中特征的数量
3. **性能提升**：准确率提升 vs 特征减少

**特征选择的价值：**
- 在保持或提升准确率的同时，减少特征数量
- 降低计算开销
- 提高模型可解释性

**学习重点：**
理解特征选择的最终目标——在性能和复杂度之间找到最佳平衡。

In [ ]:
# 全部特征
clf = KNeighborsClassifier(n_neighbors=3)
full_score = cross_val_score(clf, X, y, cv=5).mean()

# PSO选中特征
X_selected = X[:, selected_idx]
selected_score = cross_val_score(clf, X_selected, y, cv=5).mean()

print("全部13个特征的准确率:", round(full_score, 4))
print("PSO选中特征的准确率:", round(selected_score, 4))
print("PSO减少的特征数:", X.shape[1] - len(selected_idx))

## PCA 可视化 PSO 选中特征后的分布

**教学说明（本科生）：**

本节使用PCA将高维特征降到2维，可视化PSO选中特征的分类效果。

**PCA（主成分分析）：**
- 一种降维技术，保留数据中方差最大的方向
- 用于可视化高维数据
- 可以观察不同类别在降维空间中的分离程度

**学习重点：**
理解如何通过降维技术可视化高维数据，体会特征选择对类别可分性的影响。

In [ ]:
if X_selected.shape[1] == 1:
    plt.figure(figsize=(7, 4))
    for cls in np.unique(y):
        plt.scatter(
            X_selected[y == cls, 0],
            np.zeros_like(X_selected[y == cls, 0]) + cls,
            label=target_names[cls]
        )
    plt.xlabel("Selected Feature Value")
    plt.title("Only One Feature Selected")
    plt.legend()
    plt.show()
else:
    pca_selected = PCA(n_components=2)
    X_selected_pca = pca_selected.fit_transform(X_selected)

    plt.figure(figsize=(7, 5))
    for i, label in enumerate(target_names):
        plt.scatter(X_selected_pca[y == i, 0], X_selected_pca[y == i, 1], label=label)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA of PSO-Selected Features (Wine)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# 相关资料
- 遗传：https://rednuht.org/genetic_walkers/ 
- 粒子群：https://thiagodnf.github.io/pso-simulator/?utm_source=chatgpt.com、https://yashsax.github.io/Particle-Swarm-Visualizer/